# Stage 5: Bone Length Measurement (Downsampled)

Measure bone length from socket center to furthest phalanx point.
Includes both v1 (simple inside/outside) and v2 (first intersection) methods.

**Input**: `labelsSocketDetected_ds/*_socket.nii.gz`, `labelsShrunk_50_ds/*_shrunk.nii.gz`, `labels_ds/*.nii.gz`, `images_ds/*.nii.gz`  
**Output**: `labelsBoneLength_ds/*_bone_length.nii.gz`, `labelsBoneLength_v2_ds/*_bone_length_v2.nii.gz`  
**Metrics**: `metrics/bone_length_metrics_v1_ds.json`, `metrics/bone_length_metrics_v2_ds.json`

Note: All measurements are in downsampled voxel space. Final metrics will be scaled in the upsample stage.

In [ ]:
# Configuration
TARGET_DIR = "/mnt/c/users/mwild/firebase/perios/levi_data_1.6.26"

# BV/TV intensity threshold
INTENSITY_THRESHOLD = 80.0

In [ ]:
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path('.').resolve()))
from utils import (
    load_nifti, save_nifti, save_metrics, ensure_dir,
    bresenham_line_3d, VOXEL_SIZE_MM
)

In [ ]:
# Setup directories (using downsampled data)
target = Path(TARGET_DIR)
SOCKET_DIR = target / "labelsSocketDetected_ds"
PHALANX_DIR = target / "labelsShrunk_50_ds"
LABELS_DIR = target / "labels_ds"
IMAGES_DIR = target / "images_ds"
METRICS_DIR = ensure_dir(target / "metrics")

OUTPUT_DIR_V1 = ensure_dir(target / "labelsBoneLength_ds")
OUTPUT_DIR_V2 = ensure_dir(target / "labelsBoneLength_v2_ds")

print(f"Socket masks: {SOCKET_DIR}")
print(f"Phalanx masks: {PHALANX_DIR}")
print(f"Original labels: {LABELS_DIR}")
print(f"Images: {IMAGES_DIR}")
print(f"Output v1: {OUTPUT_DIR_V1}")
print(f"Output v2: {OUTPUT_DIR_V2}")

In [ ]:
def measure_bone_length_v1(socket_mask, phalanx_mask, image_data, groundtruth_mask):
    """
    V1: Measure bone length using inside/outside phalanx classification.
    """
    # Compute socket center of mass
    socket_coords = np.argwhere(socket_mask > 0)
    if len(socket_coords) == 0:
        return None, None
    
    socket_com = socket_coords.mean(axis=0)
    
    # Find furthest phalanx point
    phalanx_coords = np.argwhere(phalanx_mask > 0)
    distances = np.linalg.norm(phalanx_coords - socket_com, axis=1)
    furthest_idx = np.argmax(distances)
    furthest_point = phalanx_coords[furthest_idx]
    max_distance = distances[furthest_idx]
    
    # Create line from furthest point to socket COM
    line_points = bresenham_line_3d(furthest_point, socket_com.astype(int))
    
    # Split line into inside/outside phalanx
    inside_phalanx = []
    outside_phalanx = []
    
    for point in line_points:
        z, y, x = point
        if (0 <= z < phalanx_mask.shape[0] and
            0 <= y < phalanx_mask.shape[1] and
            0 <= x < phalanx_mask.shape[2]):
            if phalanx_mask[z, y, x] > 0:
                inside_phalanx.append(point)
            else:
                outside_phalanx.append(point)
        else:
            outside_phalanx.append(point)
    
    inside_phalanx = np.array(inside_phalanx) if inside_phalanx else np.array([]).reshape(0, 3)
    outside_phalanx = np.array(outside_phalanx) if outside_phalanx else np.array([]).reshape(0, 3)
    
    # Compute Euclidean bone length
    if len(inside_phalanx) > 1:
        segments = np.diff(inside_phalanx, axis=0)
        bone_length_euclidean = float(np.sum(np.linalg.norm(segments, axis=1)))
    else:
        bone_length_euclidean = 0.0
    
    # Create visualization
    viz_mask = np.zeros_like(phalanx_mask, dtype=np.uint8)
    viz_mask[phalanx_mask > 0] = 1
    viz_mask[socket_mask > 0] = 2
    
    for point in outside_phalanx:
        z, y, x = point
        if 0 <= z < viz_mask.shape[0] and 0 <= y < viz_mask.shape[1] and 0 <= x < viz_mask.shape[2]:
            viz_mask[z, y, x] = 3
    
    for point in inside_phalanx:
        z, y, x = point
        if 0 <= z < viz_mask.shape[0] and 0 <= y < viz_mask.shape[1] and 0 <= x < viz_mask.shape[2]:
            viz_mask[z, y, x] = 4
    
    z, y, x = furthest_point
    viz_mask[z, y, x] = 5
    
    z, y, x = socket_com.astype(int)
    if 0 <= z < viz_mask.shape[0] and 0 <= y < viz_mask.shape[1] and 0 <= x < viz_mask.shape[2]:
        viz_mask[z, y, x] = 6
    
    # Compute BV/TV
    TV_phalanx = int(np.sum(phalanx_mask))
    BV_phalanx = int(np.sum((phalanx_mask > 0) & (image_data >= INTENSITY_THRESHOLD)))
    BV_TV_phalanx = float(BV_phalanx / TV_phalanx) if TV_phalanx > 0 else 0.0
    
    TV_groundtruth = int(np.sum(groundtruth_mask))
    BV_groundtruth = int(np.sum((groundtruth_mask > 0) & (image_data >= INTENSITY_THRESHOLD)))
    BV_TV_groundtruth = float(BV_groundtruth / TV_groundtruth) if TV_groundtruth > 0 else 0.0
    
    metrics = {
        'socket_com': socket_com.tolist(),
        'furthest_point': furthest_point.tolist(),
        'euclidean_distance': float(max_distance),
        'line_length_total': len(line_points),
        'line_length_inside_phalanx': len(inside_phalanx),
        'line_length_outside_phalanx': len(outside_phalanx),
        'bone_length_voxels': int(len(inside_phalanx)),
        'bone_length_euclidean': bone_length_euclidean,
        'socket_volume': int(np.sum(socket_mask)),
        'phalanx_volume': int(np.sum(phalanx_mask)),
        'TV_phalanx': TV_phalanx,
        'BV_phalanx': BV_phalanx,
        'BV_TV_phalanx': BV_TV_phalanx,
        'TV_groundtruth': TV_groundtruth,
        'BV_groundtruth': BV_groundtruth,
        'BV_TV_groundtruth': BV_TV_groundtruth,
        'intensity_threshold': INTENSITY_THRESHOLD,
    }
    
    return viz_mask, metrics

In [ ]:
def measure_bone_length_v2(socket_mask, bone_mask, image_data, secondary_mask):
    """
    V2: Measure bone length using first intersection method.
    Line goes from socket COM to furthest point, split at first bone intersection.
    """
    # Compute socket center of mass
    socket_coords = np.argwhere(socket_mask > 0)
    if len(socket_coords) == 0:
        return None, None
    
    socket_com = socket_coords.mean(axis=0)
    
    # Find furthest bone point
    bone_coords = np.argwhere(bone_mask > 0)
    distances = np.linalg.norm(bone_coords - socket_com, axis=1)
    furthest_idx = np.argmax(distances)
    furthest_point = bone_coords[furthest_idx]
    max_distance = distances[furthest_idx]
    
    # Create line from socket COM to furthest point
    line_points = bresenham_line_3d(socket_com.astype(int), furthest_point)
    
    # Find first intersection with bone mask
    intersection_idx = None
    intersection_point = None
    
    for idx, point in enumerate(line_points):
        z, y, x = point
        if (0 <= z < bone_mask.shape[0] and
            0 <= y < bone_mask.shape[1] and
            0 <= x < bone_mask.shape[2]):
            if bone_mask[z, y, x] > 0:
                intersection_idx = idx
                intersection_point = point
                break
    
    if intersection_idx is None:
        socket_segment = line_points.copy()
        bone_segment = np.array([]).reshape(0, 3)
    else:
        socket_segment = line_points[:intersection_idx]
        bone_segment = line_points[intersection_idx:]
    
    # Compute Euclidean lengths
    if len(bone_segment) > 1:
        segments = np.diff(bone_segment, axis=0)
        bone_length_euclidean = float(np.sum(np.linalg.norm(segments, axis=1)))
    else:
        bone_length_euclidean = 0.0
    
    if len(socket_segment) > 1:
        segments = np.diff(socket_segment, axis=0)
        socket_length_euclidean = float(np.sum(np.linalg.norm(segments, axis=1)))
    else:
        socket_length_euclidean = 0.0
    
    # Create visualization
    viz_mask = np.zeros_like(bone_mask, dtype=np.uint8)
    viz_mask[bone_mask > 0] = 1
    viz_mask[socket_mask > 0] = 2
    
    for point in socket_segment:
        z, y, x = point
        if 0 <= z < viz_mask.shape[0] and 0 <= y < viz_mask.shape[1] and 0 <= x < viz_mask.shape[2]:
            viz_mask[z, y, x] = 3
    
    for point in bone_segment:
        z, y, x = point
        if 0 <= z < viz_mask.shape[0] and 0 <= y < viz_mask.shape[1] and 0 <= x < viz_mask.shape[2]:
            viz_mask[z, y, x] = 4
    
    z, y, x = furthest_point
    viz_mask[z, y, x] = 5
    
    z, y, x = socket_com.astype(int)
    if 0 <= z < viz_mask.shape[0] and 0 <= y < viz_mask.shape[1] and 0 <= x < viz_mask.shape[2]:
        viz_mask[z, y, x] = 6
    
    if intersection_point is not None:
        z, y, x = intersection_point
        if 0 <= z < viz_mask.shape[0] and 0 <= y < viz_mask.shape[1] and 0 <= x < viz_mask.shape[2]:
            viz_mask[z, y, x] = 7
    
    # Compute BV/TV
    TV_primary = int(np.sum(bone_mask))
    BV_primary = int(np.sum((bone_mask > 0) & (image_data >= INTENSITY_THRESHOLD)))
    BV_TV_primary = float(BV_primary / TV_primary) if TV_primary > 0 else 0.0
    
    TV_secondary = int(np.sum(secondary_mask))
    BV_secondary = int(np.sum((secondary_mask > 0) & (image_data >= INTENSITY_THRESHOLD)))
    BV_TV_secondary = float(BV_secondary / TV_secondary) if TV_secondary > 0 else 0.0
    
    metrics = {
        'socket_com': socket_com.tolist(),
        'furthest_point': furthest_point.tolist(),
        'first_intersection': intersection_point.tolist() if intersection_point is not None else None,
        'intersection_index': int(intersection_idx) if intersection_idx is not None else None,
        'euclidean_distance_total': float(max_distance),
        'line_length_total': len(line_points),
        'line_length_socket_segment': len(socket_segment),
        'line_length_bone_segment': len(bone_segment),
        'bone_length_voxels': int(len(bone_segment)),
        'bone_length_euclidean': bone_length_euclidean,
        'socket_length_voxels': int(len(socket_segment)),
        'socket_length_euclidean': socket_length_euclidean,
        'socket_volume': int(np.sum(socket_mask)),
        'bone_volume': int(np.sum(bone_mask)),
        'TV_primary': TV_primary,
        'BV_primary': BV_primary,
        'BV_TV_primary': BV_TV_primary,
        'TV_secondary': TV_secondary,
        'BV_secondary': BV_secondary,
        'BV_TV_secondary': BV_TV_secondary,
        'intensity_threshold': INTENSITY_THRESHOLD,
    }
    
    return viz_mask, metrics

In [ ]:
# Get socket files
socket_files = sorted(SOCKET_DIR.glob("*_socket.nii.gz"))
print(f"Found {len(socket_files)} socket files to process")
print("="*70)

all_metrics_v1 = []
all_metrics_v2 = []

for idx, socket_file in enumerate(socket_files, 1):
    sample_name = socket_file.stem.replace('_socket', '').replace('.nii', '')
    print(f"\n[{idx}/{len(socket_files)}] {sample_name}")
    
    # Find corresponding files
    phalanx_file = PHALANX_DIR / f"{sample_name}_shrunk.nii.gz"
    labels_file = LABELS_DIR / f"{sample_name}.nii.gz"
    image_file = IMAGES_DIR / f"{sample_name}.nii.gz"
    
    # Check for alternative naming patterns
    if not image_file.exists():
        # Try with _0000 suffix (original pipeline format)
        image_file = IMAGES_DIR / f"{sample_name}_0000.nii.gz"
    if not image_file.exists():
        # Try with _XXX suffix (levi data format: _000, _001, etc.)
        matches = list(IMAGES_DIR.glob(f"{sample_name}_*.nii.gz"))
        if matches:
            image_file = matches[0]
    
    missing = []
    if not phalanx_file.exists():
        missing.append(f"phalanx: {phalanx_file.name}")
    if not labels_file.exists():
        missing.append(f"labels: {labels_file.name}")
    if not image_file.exists():
        missing.append(f"image: {sample_name}*.nii.gz")
    
    if missing:
        print(f"    SKIPPED - Missing: {', '.join(missing)}")
        continue
    
    try:
        # Load all data
        socket_data, affine, header = load_nifti(socket_file)
        socket_mask = (socket_data > 0).astype(np.uint8)
        
        phalanx_data, _, _ = load_nifti(phalanx_file)
        phalanx_mask = (phalanx_data > 0).astype(np.uint8)
        
        labels_data, _, _ = load_nifti(labels_file)
        labels_mask = (labels_data > 0).astype(np.uint8)
        
        image_data, _, _ = load_nifti(image_file)
        
        # V1: Inside/outside method
        viz_v1, metrics_v1 = measure_bone_length_v1(socket_mask, phalanx_mask, image_data, labels_mask)
        if viz_v1 is not None:
            output_v1 = OUTPUT_DIR_V1 / f"{sample_name}_bone_length.nii.gz"
            save_nifti(viz_v1, affine, header, output_v1)
            metrics_v1['sample_name'] = sample_name
            metrics_v1['filename'] = socket_file.name
            all_metrics_v1.append(metrics_v1)
            print(f"    V1: Bone length = {metrics_v1['bone_length_euclidean']:.1f} voxels")
        
        # V2: First intersection method (using labels as bone mask)
        viz_v2, metrics_v2 = measure_bone_length_v2(socket_mask, labels_mask, image_data, phalanx_mask)
        if viz_v2 is not None:
            output_v2 = OUTPUT_DIR_V2 / f"{sample_name}_bone_length_v2.nii.gz"
            save_nifti(viz_v2, affine, header, output_v2)
            metrics_v2['sample_name'] = sample_name
            metrics_v2['filename'] = socket_file.name
            metrics_v2['bone_mask_type'] = 'labels'
            metrics_v2['secondary_mask_type'] = 'shrunk_50'
            all_metrics_v2.append(metrics_v2)
            print(f"    V2: Bone length = {metrics_v2['bone_length_euclidean']:.1f} voxels")
        
    except Exception as e:
        print(f"    ERROR: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# Save metrics
metrics_file_v1 = METRICS_DIR / "bone_length_metrics_v1_ds.json"
metrics_file_v2 = METRICS_DIR / "bone_length_metrics_v2_ds.json"

save_metrics(all_metrics_v1, metrics_file_v1)
save_metrics(all_metrics_v2, metrics_file_v2)

# Summary
print("\n" + "="*70)
print("BONE LENGTH MEASUREMENT COMPLETE (DOWNSAMPLED)")
print("="*70)

print(f"\nV1 (inside/outside phalanx):")
print(f"  Processed: {len(all_metrics_v1)} samples")
if all_metrics_v1:
    bone_lengths = [m['bone_length_euclidean'] for m in all_metrics_v1]
    print(f"  Bone length: {np.mean(bone_lengths):.1f} +/- {np.std(bone_lengths):.1f} voxels (downsampled)")
print(f"  Output: {OUTPUT_DIR_V1}")
print(f"  Metrics: {metrics_file_v1}")

print(f"\nV2 (first intersection):")
print(f"  Processed: {len(all_metrics_v2)} samples")
if all_metrics_v2:
    bone_lengths = [m['bone_length_euclidean'] for m in all_metrics_v2]
    print(f"  Bone length: {np.mean(bone_lengths):.1f} +/- {np.std(bone_lengths):.1f} voxels (downsampled)")
print(f"  Output: {OUTPUT_DIR_V2}")
print(f"  Metrics: {metrics_file_v2}")

print(f"\nNote: Metrics are in downsampled space. Final scaling in upsample stage.")
print(f"\nVisualization classes:")
print(f"  0 = background")
print(f"  1 = phalanx/bone")
print(f"  2 = socket")
print(f"  3 = line (outside/socket segment)")
print(f"  4 = line (inside/bone segment)")
print(f"  5 = furthest point")
print(f"  6 = socket COM")
print(f"  7 = intersection point (v2 only)")